# MODUL PRAKTIKUM BIG DATA
## Pertemuan 10 — Pemrosesan Data Streaming dengan Spark Structured Streaming

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 10 |
| **Topik** | Konsep Batch vs Streaming, Spark Structured Streaming, Agregasi Data Streaming |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-9 selesai (Spark/PySpark 3.5.9 terpasang, memahami DataFrame, join, window function) |

---

### Cara Menggunakan Modul Ini

**Tidak ada instalasi baru** pada pertemuan ini — Spark Structured Streaming adalah bagian bawaan dari Spark 3.5.9 yang sudah anda pasang sejak Pertemuan 4. Seluruh modul dikerjakan langsung di Jupyter Notebook.

> **Konsistensi versi:** Tetap **Spark/PySpark 3.5.9**, **Python 3.11** — tidak ada perubahan versi apa pun. Modul ini murni memperkenalkan API baru (`readStream`/`writeStream`) yang sudah tersedia dalam paket Spark yang sama.


---
## Recap Pertemuan Sebelumnya

- [ ] Memahami `SparkSession`, DataFrame, `groupBy`/`agg` dari Pertemuan 4-7
- [ ] Memahami MongoDB & PyMongo dari Pertemuan 9 (tidak wajib aktif hari ini)

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 10, mahasiswa mampu:
1. Menjelaskan perbedaan mendasar antara pemrosesan **batch** dan **streaming**.
2. Menjelaskan konsep dasar Spark Structured Streaming: *input source*, *trigger*, *output sink*, *output mode*.
3. Membangun sebuah *streaming query* sederhana yang membaca data secara inkremental dan melakukan agregasi.
4. Menerapkan agregasi berbasis **window waktu** pada data streaming.

---

## 10.1 Batch vs Streaming: Apa Bedanya?

Sepanjang Pertemuan 3-9, seluruh data yang kita olah bersifat **statis** — sudah lengkap tersimpan (di CSV, HDFS, atau MongoDB) sebelum kita mulai memprosesnya. Pendekatan ini disebut pemrosesan **batch**.

Di dunia nyata, banyak data justru **terus mengalir tanpa henti** — transaksi e-commerce yang masuk setiap detik, data sensor IoT, log aktivitas pengguna aplikasi, atau feed media sosial. Menunggu data "selesai lengkap" sebelum diproses tidak masuk akal untuk kasus semacam ini — kita perlu memprosesnya **secara inkremental**, sesegera mungkin setelah data tersebut tiba. Inilah yang disebut pemrosesan **streaming**.

| Aspek | Batch Processing | Stream Processing |
|---|---|---|
| **Sifat data** | Statis, sudah lengkap tersimpan | Terus mengalir/bertambah |
| **Kapan diproses** | Terjadwal (mis. setiap malam) | Sesegera data tiba (near real-time) |
| **Contoh pada praktikum kita** | Pertemuan 4-7 (baca CSV/Parquet yang sudah ada) | Pertemuan 10 (hari ini) |
| **Contoh dunia nyata** | Laporan penjualan bulanan | Deteksi transaksi mencurigakan (fraud) secara langsung |

**Spark Structured Streaming** memungkinkan kita menulis kode pemrosesan streaming menggunakan **DataFrame API yang sama persis** dengan yang sudah kita kuasai sejak Pertemuan 4 — Spark menerjemahkannya secara otomatis menjadi serangkaian pemrosesan *micro-batch* (batch berukuran sangat kecil, diproses berulang setiap beberapa saat) di belakang layar.

> **Simulasi pada praktikum ini:** Sumber data streaming yang sesungguhnya (mis. Apache Kafka, sensor IoT) membutuhkan infrastruktur tambahan yang kompleks. Untuk belajar konsepnya, kita akan **mensimulasikan aliran data menggunakan folder** — setiap kali sebuah berkas baru muncul di folder tersebut, Spark akan otomatis memprosesnya, persis seperti data yang "mengalir masuk". Ini adalah cara belajar Structured Streaming yang standar dan diakui secara luas sebelum melangkah ke sumber streaming real seperti Kafka.


---
## 10.2 Persiapan: SparkSession & Folder Simulasi Streaming

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan10-StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Mengurangi jumlah partisi shuffle default (200) agar pemrosesan pada VM/laptop kita lebih cepat
# -- semakin kecil dataset (seperti simulasi kita), semakin tidak perlu partisi sebanyak itu
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/05 07:40:10 WARN Utils: Your hostname, asfadani resolves to a loopback address: 127.0.1.1; using 10.90.70.86 instead (on interface wlx088af1bd9610)
26/11/05 07:40:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 07:40:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# Menyiapkan folder kosong yang akan berperan sebagai "sumber data streaming"
folder_stream = "simulasi_stream_transaksi"
folder_checkpoint = "checkpoint_pertemuan10"

shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"Folder simulasi streaming '{folder_stream}' siap (kosong).")

Folder simulasi streaming 'simulasi_stream_transaksi' siap (kosong).


> **Wajib mendefinisikan skema secara eksplisit.** Berbeda dengan pembacaan batch pada Pertemuan 4 (`inferSchema=True`), Spark Structured Streaming **tidak mengizinkan** penebakan skema otomatis — karena Spark tidak bisa "menunggu semua data selesai" untuk menebak tipe datanya (**datanya memang belum semuanya ada**). Skema harus didefinisikan secara eksplisit menggunakan `StructType`.

In [3]:
# Mendefinisikan skema secara eksplisit — wajib untuk streaming source berbasis berkas
skema_transaksi = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

print("Skema berhasil didefinisikan:")
print(skema_transaksi)

Skema berhasil didefinisikan:
StructType([StructField('trx_id', StringType(), True), StructField('kota', StringType(), True), StructField('kategori', StringType(), True), StructField('pendapatan', IntegerType(), True)])


---
## 10.3 Membangun Streaming Query Pertama

### Tahap 1: Mendefinisikan Input Source (`readStream`)

In [5]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_transaksi) \
    .option("header", True) \
    .load("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/simulasi_stream_transaksi")

print("Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').")
print("df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').
df_stream.isStreaming: True


> Perhatikan `df_stream.isStreaming` bernilai `True` — inilah yang membedakan DataFrame streaming dari DataFrame batch biasa yang sudah sering kita pakai. Sama seperti *lazy evaluation* pada Pertemuan 4, mendefinisikan `df_stream` **belum** memulai pemrosesan apa pun.

### Tahap 2: Mendefinisikan Transformasi (Sama Persis Seperti DataFrame Biasa!)

In [6]:
# Melakukan agregasi — sintaks IDENTIK dengan groupBy/agg pada DataFrame batch (Pertemuan 4-5)
agregasi_stream = df_stream.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

print("Transformasi berhasil didefinisikan.")

Transformasi berhasil didefinisikan.


### Tahap 3: Mendefinisikan Output Sink (`writeStream`) & Memulai Query

Kita akan menggunakan **`memory` sink** — hasil agregasi disimpan sementara sebagai tabel in-memory bernama `hasil_agregasi`, yang dapat kita kueri kapan saja menggunakan `spark.sql()`. Sink ini cocok untuk **belajar & debugging**; pada sistem produksi biasanya dipakai sink seperti file/Kafka/database.

In [7]:
query = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_agregasi") \
    .start()

print("Streaming query dimulai! Status:", query.status)

Streaming query dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


> **`outputMode("complete")`** berarti setiap kali ada data baru, Spark menampilkan **ulang keseluruhan** hasil agregasi terbaru (bukan hanya baris yang berubah) — mode paling sederhana untuk dipahami pemula, cocok untuk agregasi seperti `groupBy` di atas.

> **Catatan penting soal waktu:** Streaming query butuh sesaat untuk "pemanasan" (*startup*) sebelum benar-benar mulai memproses data pertamanya — ini normal dan terjadi pada seluruh sistem streaming real (bukan hanya Spark). Jika tabel hasil pada cell berikutnya tampil **kosong**, itu bukan berarti error — cukup tunggu beberapa detik lagi lalu jalankan ulang cell `spark.sql(...)` yang menampilkan hasilnya.

### Tahap 4: Mengalirkan "Data Baru" dan Mengamati Hasilnya

Sekarang mari simulasikan data yang "mengalir masuk" dengan menuliskan berkas CSV baru ke folder yang sedang diawasi Spark.

In [8]:
# Batch data pertama "tiba"
pd.DataFrame({
    "trx_id": ["T1", "T2", "T3"],
    "kota": ["Magelang", "Solo", "Magelang"],
    "kategori": ["Elektronik", "Fashion", "Makanan"],
    "pendapatan": [150000, 200000, 75000],
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)  # memberi waktu Spark memproses micro-batch (proses pertama kali butuh waktu ekstra)

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+--------+----------------+----------------+
|    kota|total_pendapatan|jumlah_transaksi|
+--------+----------------+----------------+
|Magelang|          225000|               2|
|    Solo|          200000|               1|
+--------+----------------+----------------+



In [9]:
# Batch data kedua "tiba" beberapa saat kemudian
pd.DataFrame({
    "trx_id": ["T4", "T5"],
    "kota": ["Magelang", "Yogyakarta"],
    "kategori": ["Fashion", "Elektronik"],
    "pendapatan": [300000, 500000],
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

Batch 2 dikirim. Menunggu Spark memproses...
+----------+----------------+----------------+
|      kota|total_pendapatan|jumlah_transaksi|
+----------+----------------+----------------+
|  Magelang|          525000|               3|
|Yogyakarta|          500000|               1|
|      Solo|          200000|               1|
+----------+----------------+----------------+



Perhatikan **`total_pendapatan` untuk Magelang otomatis bertambah** setelah batch 2 masuk (dari batch 1 + batch 2), **tanpa kita menulis ulang kode apa pun** — Spark secara otomatis mempertahankan **state** (kondisi akumulasi) dari hasil agregasi sebelumnya dan menggabungkannya dengan data baru. Inilah kekuatan utama Structured Streaming.

### Tahap 5: Menghentikan Streaming Query

Streaming query akan **terus berjalan tanpa henti** hingga dihentikan secara eksplisit (berbeda dengan query batch biasa yang otomatis selesai). Selalu hentikan query yang sedang berjalan sebelum melanjutkan ke bagian lain notebook.

In [10]:
query.stop()
print("Streaming query dihentikan. Status aktif:", query.isActive)

Streaming query dihentikan. Status aktif: False


---
## 10.4 Agregasi Berbasis Window Waktu

Selain mengelompokkan berdasarkan kolom biasa (seperti `kota` di atas), Structured Streaming juga mendukung pengelompokan berdasarkan **rentang/window waktu** — sangat berguna untuk pertanyaan seperti *"berapa total transaksi setiap 1 menit?"*. Ini **berbeda** dari window function pada Pertemuan 5 (`rank`/`row_number` untuk perankingan) — window di sini merujuk pada **rentang waktu**, bukan pengelompokan data untuk perankingan.

Kali ini kita membutuhkan kolom bertipe **timestamp** agar Spark tahu "kapan" setiap data terjadi.

In [13]:
skema_dengan_waktu = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
    StructField("waktu_transaksi", TimestampType(), True),
])

# Membuat ulang folder simulasi (kosong) untuk demo window waktu
folder_stream2 = "simulasi_stream_waktu"
shutil.rmtree(folder_stream2, ignore_errors=True)
os.makedirs(folder_stream2, exist_ok=True)

df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_dengan_waktu) \
    .option("header", True) \
    .load("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/simulasi_stream_waktu")

# Mengelompokkan berdasarkan window waktu 1 menit, dihitung dari kolom waktu_transaksi
agregasi_window = df_stream2.groupBy(
    window(col("waktu_transaksi"), "1 minute")
).agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_window") \
    .start()

print("Streaming query window waktu dimulai.")

Streaming query window waktu dimulai.


In [14]:
from datetime import datetime

# Mengirim data dengan timestamp saat ini
waktu_sekarang = datetime.now()
pd.DataFrame({
    "trx_id": ["W1", "W2"],
    "kota": ["Semarang", "Solo"],
    "pendapatan": [120000, 180000],
    "waktu_transaksi": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2,
}).to_csv(f"{folder_stream2}/batch_w1.csv", index=False)

time.sleep(6)
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_transaksi FROM hasil_window").show(truncate=False)

+-------------------+-------------------+----------------+----------------+
|start              |end                |total_pendapatan|jumlah_transaksi|
+-------------------+-------------------+----------------+----------------+
|2026-11-05 08:20:00|2026-11-05 08:21:00|300000          |2               |
+-------------------+-------------------+----------------+----------------+



Perhatikan hasil kueri menampilkan kolom `window` berisi rentang waktu awal (`start`) dan akhir (`end`) selebar 1 menit — seluruh transaksi yang `waktu_transaksi`-nya jatuh dalam rentang yang sama otomatis dikelompokkan dan dijumlahkan bersama. Konsep inilah yang mendasari dashboard *real-time* seperti "jumlah pesanan per menit" yang sering kalian lihat pada aplikasi e-commerce.

In [15]:
query2.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` (Sub-bab 10.2) sebelum mengerjakan latihan berikut.

In [16]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_latihan = "latihan10_stream"
shutil.rmtree(folder_latihan, ignore_errors=True)
os.makedirs(folder_latihan, exist_ok=True)

skema_latihan = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])
print("Siap untuk latihan. Folder:", folder_latihan)

Siap untuk latihan. Folder: latihan10_stream


**Soal 1.** Definisikan `readStream` dari folder `latihan10_stream` menggunakan `skema_latihan`, lalu buat agregasi `groupBy("kategori")` yang menghitung **rata-rata** `pendapatan` (gunakan `avg`, bukan `sum`). Jalankan query dengan `queryName="latihan_agg"`.

In [19]:
# Jawaban Soal 1 di sini

# Mendefinisikan readStream
df_stream_latihan = spark.readStream \
    .format("csv") \
    .schema(skema_latihan) \
    .option("header", True) \
    .load("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/latihan10_stream")

print("Streaming DataFrame berhasil didefinisikan (belum berjalan - baru 'rencana').")
print("df_stream_latihan.isStreaming:", df_stream_latihan.isStreaming)

# Membuat transform aggregasi yang menghitung rata-rata
agg_latihan = df_stream_latihan.groupBy("kategori").agg(
    avg("pendapatan").alias("avg_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

# Mendefinisikan output
query_latihan = agg_latihan.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("latihan_agg")\
    .start()

print("Streaming query dimulai! Status:", query_latihan.status)


Streaming DataFrame berhasil didefinisikan (belum berjalan - baru 'rencana').
df_stream_latihan.isStreaming: True
Streaming query dimulai! Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


**Soal 2.** Kirim (tulis) sebuah berkas CSV berisi minimal 3 baris data ke folder `latihan10_stream`, tunggu beberapa detik, lalu tampilkan isi tabel `latihan_agg` menggunakan `spark.sql()`.

In [21]:
# Jawaban Soal 2 di sini
pd.DataFrame({
    "trx_id": ["T1", "T2", "T3"],
    "kategori": ["Elektronik", "Fashion", "Makanan"],
    "pendapatan": [300000, 150000, 50000]
}).to_csv(f"{folder_latihan}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)  # memberi waktu Spark memproses micro-batch (proses pertama kali butuh waktu ekstra)

spark.sql("SELECT * FROM latihan_agg ORDER BY avg_pendapatan DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...
+----------+--------------+----------------+
|  kategori|avg_pendapatan|jumlah_transaksi|
+----------+--------------+----------------+
|Elektronik|      300000.0|               1|
|   Fashion|      150000.0|               1|
|   Makanan|       50000.0|               1|
+----------+--------------+----------------+



**Soal 3.** Kirim **batch kedua** dengan kategori yang **sama** seperti salah satu kategori pada batch pertama, tunggu beberapa detik, lalu tampilkan ulang `latihan_agg`. Amati: apakah nilai rata-ratanya berubah sesuai perhitungan gabungan kedua batch?

In [22]:
# Jawaban Soal 3 di sini
pd.DataFrame({
    "trx_id": ["T4", "T5", "T6"],
    "kategori": ["Makanan", "Fashion", "Makanan"],
    "pendapatan": [30000, 200000, 25000]
}).to_csv(f"{folder_latihan}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)  # memberi waktu Spark memproses micro-batch (proses pertama kali butuh waktu ekstra)

spark.sql("SELECT * FROM latihan_agg ORDER BY avg_pendapatan DESC").show()

Batch 2 dikirim. Menunggu Spark memproses...
+----------+--------------+----------------+
|  kategori|avg_pendapatan|jumlah_transaksi|
+----------+--------------+----------------+
|Elektronik|      300000.0|               1|
|   Fashion|      175000.0|               2|
|   Makanan|       35000.0|               3|
+----------+--------------+----------------+



In [23]:
query_latihan.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa Spark Structured Streaming **mewajibkan** skema didefinisikan secara eksplisit (`StructType`), sementara pemrosesan batch pada Pertemuan 4 bisa memakai `inferSchema=True`? Tulis jawaban pada markdown cell di bawah ini.

*(Tulis jawaban di sini — jangan lupa `query.stop()` dan `spark.stop()` setelah selesai berlatih)*
>Karena pada pemrosesan secara batch data sudah ada/masuk secara lengkap, sehingga spark bisa melihat keseluruhan data untuk menebak tipe datanya. Berbeda dengan proses stream, dimana datanya awalnya belum ada sehingga jika tidak didefinisikan di awal bisa menurunkan performa proses analisisnya dimana spark harus menebak tipe data juga selain melakukan kueri. Selain hal itu juga dapat menjaga kualitas data yang masuk agar tidak ada noise (noise ditangani).

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas10_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan seluruh streaming query dihentikan (`query.stop()`) di akhir notebook sebelum dikumpulkan.
- Seluruh cell wajib sudah dijalankan (**Run All**) — perhatikan penggunaan `time.sleep()` yang cukup agar output streaming sempat terisi sebelum notebook mencoba menampilkannya.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Streaming Source | Skema & `readStream` didefinisikan dengan benar | 15% |
| B. Agregasi per Cabang | Kolom turunan & agregasi `groupBy` benar, query berjalan | 20% |
| C. Window Waktu | Agregasi window 2 menit terdefinisi & berjalan dengan benar | 20% |
| D. Pengujian Multi-Batch | Minimal 3 batch terkirim, hasil akumulasi terbukti benar | 25% |
| E. File Sink | Streaming ke Parquet berhasil, terverifikasi dapat dibaca kembali | 20% |
